## Final UP5 revision experiments
Run the setup cell first from the notebooks directory. It displays saved results without fitting models. The following four cells reproduce individual experiments. For a new benchmark, set benchmark_run to its output directory before running dependent cells. The native-token experiment saves a separate run. TabPFN fitting requires its pretrained weights.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

src = str(Path("../src").resolve())
if src not in sys.path:
    sys.path.insert(0, src)
data_dir = Path("../data/UP5")
benchmark_run = Path("../outputs/up5_revision/20261101-112858")
native_run = Path("../outputs/up5_native_ablation/20261101-134759-437521")
for result in (benchmark_run / "scores.csv", benchmark_run / "mdn-backbones.csv", native_run / "comparison.csv"):
    print(result.name)
    display(pd.read_csv(result))

In [ ]:
from experiments.up5.target_only import run_benchmark
from pathlib import Path
from datetime import datetime
output = Path("../outputs/up5_revision") / datetime.now().strftime("%Y%m%d-%H%M%S")
run_benchmark(data_dir, output, tabpfn_kwargs={})

In [ ]:
# After setup: scores the completed run without retraining FT.
from experiments.up5.distribution import compare
import pandas as pd
pd.DataFrame(compare(data_dir, benchmark_run)).T

In [ ]:
# Revision: MLP-MDN backbone comparison. Run after setup.
from experiments.up5.encoded_backbones import compare_backbones
compare_backbones(data_dir, benchmark_run)

In [ ]:
# Revision: native-token Transformer versus pooled MLP (fits both models).
from experiments.up5.native_backbones import compare_native
compare_native(data_dir, "../outputs/up5_native_ablation")

## MDN Head Ablation UP5

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import up5_helper
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# 1) Load data once
X_train, X_test, y_train, y_test = up5_helper.load_dataset("../data/UP5/", only_36=False)
X_train = up5_helper.reconstruct_categories(X_train)
X_test  = up5_helper.reconstruct_categories(X_test)
all_cols = list(X_train.columns)

# 2) Base config: early_stopping=True now
common_config = {
    'epochs': 80,
    'deterministic': True,
    'benchmark': False,
    'verbose': False,
    'early_stopping': True,  
    'patience': 50,     
    'd_token': 56,
    'n_heads': 4,
    'n_blocks': 4,
    'ff_factor': 8.0,
    'dropout': 0.15,
    'learning_rate': 3e-4,
    'weight_decay': 5e-5,
    'mdn_n_components': 2,
    'mdn_temperature': 2.5,
    'mdn_hidden_size': 24,
    'batch_size': 32,
    'use_interaction': True,
    'use_scheduler': False
}

In [ ]:
from models.ft_mdn_transformer import FTTransformerModel

model = FTTransformerModel(
    sample_df=X_train,
    seed=42,
    #shared_num_cols=shared_num_cols,  # alias 'shared_num' also accepted
    #shared_cat_cols=shared_cat_cols,  # alias 'shared_cat' also accepted
    use_mdn=True,
    epochs=80, d_token=56, n_heads=4, n_blocks=4, ff_factor=8.0,
    dropout=0.15, learning_rate=3e-4, weight_decay=5e-5,
    mdn_n_components=2, mdn_temperature=2.5, mdn_hidden_size=24,
    batch_size=32, use_interaction=True, use_scheduler=False
)

model.train(X_train[all_cols], y_train, X_test[all_cols], y_test)
model.select_best_model()

# Point preds when you need them
y_pred = model.predict(X_test[all_cols])


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_portfolio_density_minimal(
    model, X_test, y_test,
    y_grid=None, grid_max=1.1, bins=30,
    savepath="../figures/up5_ft_mdn_density.pdf",
    title="Portfolio Predictive Density (Mixture-of-Mixtures)"
):
    # Colors consistent with your per-loan plot
    mix_color  = "#00376c"   # dark blue (predicted mixture)
    hist_color = "#b7d6f0"   # very light blue (empirical histogram)

    # 1) Predicted portfolio density
    if y_grid is None:
        y_grid = np.linspace(0.0, grid_max, 600)
    y_grid, pred_density = model.portfolio_density(X_test, y_grid=y_grid)

    # 2) Plot
    plt.rcParams["font.family"] = "Times New Roman"
    fig, ax = plt.subplots(figsize=(7.4, 4.2))

    # Empirical histogram of test set
    ax.hist(y_test, bins=bins, density=True, color=hist_color, edgecolor="none",
            alpha=0.7, label="Test empirical (hist)", zorder=2)

    # Predicted mixture-of-mixtures density
    ax.plot(y_grid, pred_density, color=mix_color, linewidth=2.6,
            label="Predicted portfolio density", zorder=3)

    # Cosmetics
    ax.set_xlim(y_grid.min(), y_grid.max())
    ax.set_xlabel("Recovery rate")
    ax.set_ylabel("Density")
    ax.set_title(title)
    ax.grid(True, alpha=0.25)

    # Legend at top center
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.04),
              ncol=2, frameon=True, fontsize=9)

    # Leave room for the top legend
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    #plt.savefig(savepath)
    plt.show()

# Example:
# y_grid = np.linspace(0, 1.1, 500)
# plot_portfolio_density_minimal(model, X_test[all_cols], y_test, y_grid=y_grid,
#                                savepath="../figures/up5_ft_mdn_density.pdf")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_portfolio_density_side_by_side(
    model,
    X_train, y_train,
    X_test,  y_test,
    *,
    mode: str = "both",          # "train", "test", or "both"
    y_grid=None, grid_max=1.1,   # grid for the predicted density
    bins=30,
    font_family="Times New Roman",
    title="Portfolio Predictive Recovery Distribution (MDN)",
    savepath="../figures/up5_ft_mdn_density_side_by_side.pdf"
):
    """
    Side-by-side portfolio density plots:
      - Left: Train (if mode includes train)
      - Right: Test (if mode includes test)

    Design:
      • Predicted mixture-of-mixtures density: dark blue line.
      • Empirical histogram: pastel (green for Train, blue for Test).
      • No means, no KDE.
      • Legend top-center inside each panel.
    """
    assert mode in {"train", "test", "both"}

    # Colors (harmonized with your per-loan plot)
    mix_color   = "#00376c"  # dark blue predicted density
    train_color = "#c9e6c9"  # light green train histogram
    test_color  = "#b7d6f0"  # light blue test histogram

    # Build a common grid if not provided
    if y_grid is None:
        y_grid = np.linspace(0.0, grid_max, 600)

    # Decide panels
    panels = []
    if mode in ("train", "both"):
        panels.append(("Train", X_train, y_train, train_color))
    if mode in ("test", "both"):
        panels.append(("Test",  X_test,  y_test,  test_color))

    # Matplotlib config
    plt.rcParams["font.family"] = font_family
    ncols = len(panels)
    fig, axes = plt.subplots(1, ncols, figsize=(6.4 * ncols / 1.3, 5.2), sharey=True)
    if ncols == 1:
        axes = [axes]

    # Draw each panel
    for ax, (name, Xp, yp, hist_color) in zip(axes, panels):
        # Predicted portfolio density on the given split
        _, pred_density = model.portfolio_density(Xp, y_grid=y_grid)

        # Empirical histogram
        ax.hist(yp, bins=bins, density=True, color=hist_color, edgecolor="none",
                alpha=0.75, label=f"{name} Set", zorder=2)

        # Predicted density
        ax.plot(y_grid, pred_density, color=mix_color, linewidth=2.6,
                label="Predicted Portfolio Density", zorder=3)

        # Cosmetics
        ax.set_xlim(y_grid.min(), y_grid.max())
        ax.set_xlabel("Recovery Rate")
        ax.set_ylabel("Density" if ax is axes[0] else "")
        ax.set_title(f"{name}")

        ax.grid(True, alpha=0.25)

        # Legend at top-center inside the axes; keep it inside with a slightly lower anchor
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles, labels,
                  loc="upper center", bbox_to_anchor=(0.5, 0.97),
                  ncol=2, frameon=True, fontsize=10)

    # Overall title and layout
    fig.suptitle(title, y=0.96, fontsize=12)
    plt.subplots_adjust(wspace=0.25, top=0.88)      # room for legends + suptitle
    plt.tight_layout(rect=[0, 0, 1, 0.93])          # keep legends inside

    if savepath:
        plt.savefig(savepath)
    plt.show()



In [ ]:
# =========================
# Example calls
# =========================
# 1) Both side-by-side (Train left, Test right)
plot_portfolio_density_side_by_side(
    model,
    X_train[all_cols], y_train,
    X_test[all_cols],  y_test,
    mode="both",
    y_grid=np.linspace(0, 1.1, 600),
    bins=30,
    savepath="../figures/up5_ft_mdn_density_side_by_side.pdf"
)

# 2) Only Train
# plot_portfolio_density_side_by_side(model, X_train[all_cols], y_train, X_test[all_cols], y_test,
#                                     mode="train", y_grid=np.linspace(0, 1.1, 600),
#                                     savepath="../figures/up5_ft_mdn_density_train.pdf")

# 3) Only Test
# plot_portfolio_density_side_by_side(model, X_train[all_cols], y_train, X_test[all_cols], y_test,
#                                     mode="test", y_grid=np.linspace(0, 1.1, 600),
#                                     savepath="../figures/up5_ft_mdn_density_test.pdf")
    

In [ ]:
# Example call (matches your snippet)
y_grid = np.linspace(0, 1.1, 500)
plot_portfolio_density_minimal(model, X_test[all_cols], y_test, y_grid=y_grid,
                               savepath="../figures/up5_ft_mdn_density.pdf")



In [ ]:
# Example call (matches your snippet)
y_grid = np.linspace(0, 1.1, 500)
plot_portfolio_density_styled(model, X_train[all_cols], y_train, y_grid=y_grid,
                               savepath="../figures/up5_ft_mdn_density.pdf")



In [ ]:
# 1) Get portfolio density predicted by the MDN as mixture-of-mixtures
y_grid, pred_density = model.portfolio_density(X_test[all_cols], y_grid=np.linspace(0, 1.1, 500))
#y_grid, pred_density = portfolio_density(model, X_test[all_cols])#, y_grid=np.linspace(0, 1.2, 500))

# 2) Plot against the empirical test distribution
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Times New Roman"

plt.figure(figsize=(7,4))
plt.hist(y_test, bins=30, density=True, alpha=0.4, label='True (test)')
plt.plot(y_grid, pred_density, label='Predicted Portfolio Density')
plt.xlabel('Recovery rate'); plt.ylabel('Density'); plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig("../figures/up5_ft_mdn_density.pdf")
plt.show()


In [ ]:
# 1) Get portfolio density predicted by the MDN as mixture-of-mixtures
y_grid, pred_density = model.portfolio_density(X_test[all_cols], y_grid=np.linspace(0, 1.1, 500))
#y_grid, pred_density = portfolio_density(model, X_train[all_cols])

# 2) Plot against the empirical test distribution
import matplotlib.pyplot as plt
plt.figure(figsize=(7,4))
plt.hist(y_train, bins=40, density=True, alpha=0.4, label='True (train)')
plt.plot(y_grid, pred_density, label='Predicted portfolio density')
plt.xlabel('Recovery rate'); plt.ylabel('Density'); plt.legend(); plt.grid(True); plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import math

def _beta_pdf(y, a, b):
    y = np.clip(y, 1e-9, 1 - 1e-9)
    lgamma = np.vectorize(math.lgamma)
    logB = lgamma(a) + lgamma(b) - lgamma(a + b)
    return np.exp((a - 1) * np.log(y) + (b - 1) * np.log(1 - y) - logB)

def _tint(hex_color, amount=0.35):
    """Lighten color toward white by `amount` (0..1)."""
    rgb = np.array(mcolors.to_rgb(hex_color))
    mixed = (1 - amount) * rgb + amount * np.ones(3)
    return mcolors.to_hex(mixed)

def plot_mdn_for_sample_styled(model, X, y_true, idx, y_min=0.0, y_max=1.2, title=None):
    """
    Component PDFs (similar hues) + mixture PDF (background) + vertical lines
    for predicted mean and truth (prominent).
    """
    assert getattr(model, "use_mdn", False), "Model must use MDN head."

    # --- extract one row & params
    row = X.iloc[[idx]]
    dist = model.predict_distribution(row)
    mdn_type = getattr(model, "mdn_distribution", "gaussian")
    pred_mean = float(dist["mean"][0])
    truth = float(y_true[idx])

    # --- grid
    if mdn_type == "gaussian":
        mu = dist["mu"][0]; sigma = dist["sigma"][0]
        lo = max(y_min, float((mu - 4 * sigma).min()))
        hi = min(y_max, float((mu + 4 * sigma).max())) if y_max is not None else float((mu + 4 * sigma).max())
        lo = min(lo, y_min); hi = max(hi, y_max)
        y = np.linspace(lo, hi, 800)
    else:  # beta on [0,1]
        lo, hi = max(0.0, y_min), min(1.0, y_max if y_max is not None else 1.0)
        y = np.linspace(max(1e-6, lo), min(hi - 1e-6, 1 - 1e-6), 800)

    # --- component & mixture PDFs
    alpha = dist["alpha"][0]
    K = len(alpha)
    comp_pdfs = []
    if mdn_type == "gaussian":
        mu = dist["mu"][0]; sigma = dist["sigma"][0]
        for k in range(K):
            pdf_k = alpha[k] * (1.0 / (sigma[k] * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((y - mu[k]) / sigma[k])**2)
            comp_pdfs.append(pdf_k)
        comp_means = mu
    else:
        a = dist["a"][0]; b = dist["b"][0]
        for k in range(K):
            pdf_k = alpha[k] * _beta_pdf(y, a[k], b[k])
            comp_pdfs.append(pdf_k)
        comp_means = a / (a + b)
    mixture = np.sum(comp_pdfs, axis=0)

    # --- palette (colorblind-friendly / Okabe–Ito inspired)
    mix_color   = "#00376c"                  # solid black for mixture
    comp_colors = [_tint("#5b89b0", 0.3),  # still use blueish shades for components
                   _tint("#5b89b0", 0.45),
                   _tint("#5b89b0", 0.7)]
    pred_color  = "#EF7C00"                # orange for predicted mean
    truth_color = "#389647"                # green for truth

    fig, ax = plt.subplots(figsize=(7.4, 4.4))

    # Mixture behind everything: thick solid black, no transparency
    ax.plot(y, mixture+0.04, color=mix_color, linewidth=2.5, alpha=1.0,
            zorder=1, label="Mixture PDF")

    # Components on top: dashed, a little thinner; subtle fill to separate visually
    for k, pdf_k in enumerate(comp_pdfs):
        color = comp_colors[k % len(comp_colors)]
        ax.fill_between(y, 0, pdf_k, color=color, alpha=0.12, zorder=2)
        ax.plot(y, pdf_k, color=color, linestyle="--", linewidth=2, zorder=3,
                label=f"Component {k+1} (α={alpha[k]:.2f})")
        ax.axvline(float(comp_means[k]), color=color, linestyle=":", linewidth=1.8, zorder=4)

    # Predicted mean & Truth: most prominent
    ax.axvline(pred_mean,  color=pred_color,  linewidth=3, zorder=6, label=f"Predicted mean = {pred_mean:.3f}")
    ax.axvline(truth,      color=truth_color, linewidth=3, linestyle="-.", zorder=6, label=f"Truth = {truth:.3f}")

    # Cosmetics
    ax.set_xlim(y.min(), y.max())
    ax.set_xlabel("Recovery rate")
    ax.set_ylabel("Density")
    ax.grid(True, alpha=0.25)
    ax.set_title(title or f"Sample idx={idx}: components vs mixture, mean & truth, UP5 target only")

    # Legend order: mixture, components, predicted, truth
    handles, labels = ax.get_legend_handles_labels()
    order = ( [i for i,l in enumerate(labels) if l.startswith("Mixture")]
            + [i for i,l in enumerate(labels) if l.startswith("Component")]
            + [i for i,l in enumerate(labels) if l.startswith("Predicted")]
            + [i for i,l in enumerate(labels) if l.startswith("Truth")] )
    ax.legend([handles[i] for i in order], [labels[i] for i in order],
              fontsize=9, frameon=True, loc="best")
    plt.tight_layout()
    plt.savefig(f"../figures/mdn_dist_sample_{idx}.pdf")
    plt.show()

# Example usage:
# err_idx = int(np.argmax(np.abs(model.predict(X_test[all_cols]) - y_test)))
# plot_mdn_for_sample_styled(model, X_test[all_cols], y_test, idx=err_idx, y_min=0.0, y_max=1.2)


In [ ]:
err_idx = 54

In [ ]:
# =========================
# Example usage
# =========================
# Pick an index to inspect (e.g., a random one or the worst error)
# err_idx = int(np.argmax(np.abs(model.predict(X_test[all_cols]) - y_test)))
#err_idx = err_idx +1

plot_mdn_for_sample_styled(
    model,
    X_test[all_cols],
    y_test,
    idx=err_idx,
    y_max=1.2  # set to 1.2 if your domain allows slightly >1; otherwise omit
)

In [ ]:
from models.ft_transformer_mdn_transfer5_RegressionHeadAblation import FTTransformerModel


common_config = {
    'epochs': 80,
    'deterministic': True,
    'benchmark': False,
    'verbose': False,
    'early_stopping': True,  
    'patience': 50,     
    'd_token': 56,
    'n_heads': 4,
    'n_blocks': 4,
    'ff_factor': 8.0,
    'dropout': 0.15,
    'learning_rate': 3e-4,
    'weight_decay': 5e-5,
    'mdn_n_components': 2,
    'mdn_temperature': 2.5,
    'mdn_hidden_size': 24,
    'batch_size': 32,
    'use_interaction': True,
    'use_scheduler': False,
    'use_mdn': True
}

model = FTTransformerModel(
                sample_df = X_train, # to get to know the categories for FT
                seed      = 42,
                shared_num=shared_num_cols, shared_cat=shared_cat_cols # in heterogeneous feature setting, put in the column names, that are shared.
                **common_config
            )
model.train(X_sub[all_cols], y_sub, X_test[all_cols], y_test)

model.fine_tune(
    X_new_train=Xtr[full_cols],  y_new_train=ytr,
    X_new_val  =Xte[full_cols],  y_new_val =yte,
    additional_epochs=50,
    freeze_backbone=False, freeze_shared=False
)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import up5_helper
from models.ft_transformer_mdn_transfer5_RegressionHeadAblation import FTTransformerModel
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# 1) Load data once
X_train, X_test, y_train, y_test = up5_helper.load_dataset("../data/UP5/", only_36=False)
X_train = up5_helper.reconstruct_categories(X_train)
X_test  = up5_helper.reconstruct_categories(X_test)
all_cols = list(X_train.columns)

# 2) Base config: early_stopping=True now
common_config = {
    'epochs': 80,
    'deterministic': True,
    'benchmark': False,
    'verbose': False,
    'early_stopping': True,  
    'patience': 50,     
    'd_token': 56,
    'n_heads': 4,
    'n_blocks': 4,
    'ff_factor': 8.0,
    'dropout': 0.15,
    'learning_rate': 3e-4,
    'weight_decay': 5e-5,
    'mdn_n_components': 2,
    'mdn_temperature': 2.5,
    'mdn_hidden_size': 24,
    'batch_size': 32,
    'use_interaction': True,
    'use_scheduler': False
}

In [ ]:


# 3) Experiment parameters
sample_sizes = [50, 100, 200, 500, 800, 1_000, len(X_train)]
heads        = ["regression", "mdn"]
n_runs       = 4

# make output dirs
os.makedirs("figures/histograms", exist_ok=True)

# 4) Collect per‑run results
records = []

for seed in range(n_runs):
    for n in sample_sizes:
        print("=====================================\n")
        for head in heads:
            print(f"Head={head}, N={n}, Seed={seed}")
            # sample subset
            rng   = np.random.RandomState(seed)
            idx   = rng.choice(len(X_train), size=n, replace=False)
            X_sub = X_train.iloc[idx]
            y_sub = y_train[idx]

            # train once
            model = FTTransformerModel(
                sample_df = X_sub,
                seed      = seed,
                use_mdn   = (head == "mdn"),
                **common_config
            )
            model.train(X_sub[all_cols], y_sub, X_test[all_cols], y_test)

            # --- final‐epoch predictions
            y_tr_final = model.predict(X_sub)
            y_te_final = model.predict(X_test)

            # --- best‐epoch predictions
            model.select_best_model()
            y_tr_best = model.predict(X_sub)
            y_te_best = model.predict(X_test)

            # compute MSE/RMSE/MAE/R²
            def metrics(y_true, y_pred):
                mse  = mean_squared_error(y_true, y_pred)
                rmse = np.sqrt(mse)
                mae  = mean_absolute_error(y_true, y_pred)
                r2   = r2_score(y_true, y_pred)
                return mse, rmse, mae, r2

            mse_te_fin, rmse_te_fin, mae_te_fin, r2_te_fin = metrics(y_test, y_te_final)
            mse_te_bst, rmse_te_bst, mae_te_bst, r2_te_bst = metrics(y_test, y_te_best)

            records.append({
                'head': head,
                'sample_size': n,
                'seed': seed,
                'mse_te_final': mse_te_fin,
                'rmse_te_final': rmse_te_fin,
                'mae_te_final': mae_te_fin,
                'r2_te_final': r2_te_fin,
                'mse_te_best': mse_te_bst,
                'rmse_te_best': rmse_te_bst,
                'mae_te_best': mae_te_bst,
                'r2_te_best': r2_te_bst,
            })

            # save histogram PDF of best model
            pdf = model._plot_histogram(
                y_train=y_sub,
                y_train_pred=y_tr_best,
                y_val=y_test,
                y_val_pred=y_te_best,
                epoch=None
            )
            #os.rename(pdf, f"figures/histograms/{head}_n{n}_seed{seed}.pdf")

# 5) Aggregate over seeds
df = pd.DataFrame.from_records(records)

agg = df.groupby(['head','sample_size']).agg([
    ('mean','mean'), ('std','std')
]).reset_index()

# flatten columns
agg.columns = [
    f"{metric}_{stat}"
    for metric, stat in agg.columns
]

# 6) Save and display
#os.makedirs("figures/summary", exist_ok=True)
#df.to_csv("figures/summary/all_runs.csv", index=False)
#agg.to_csv("figures/summary/metrics_summary.csv", index=False)

print("Per‐seed results → figures/summary/all_runs.csv")
print("Aggregated mean±std → figures/summary/metrics_summary.csv")


In [ ]:
all_runs

In [ ]:
agg[["head_", "sample_size_", "r2_te_final_mean", "r2_te_best_mean"]]

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the aggregated metrics (mean ± std) across seeds
#agg = pd.read_csv("figures/summary/metrics_summary.csv")

# We'll focus on test-set "best epoch" metrics: rmse_te_best, mae_te_best, r2_te_best
metrics = [
    ("rmse_te_best_mean", "rmse_te_best_std", "RMSE"),
    ("mae_te_best_mean",  "mae_te_best_std",  "MAE"),
    ("r2_te_best_mean",   "r2_te_best_std",   "R²"),
]

heads = agg['head_'].unique()
sample_sizes = sorted(agg['sample_size_'].unique())

# Create line plots with error bars
for mean_col, std_col, label in metrics:
    plt.figure(figsize=(6, 4))
    for head in heads:
        dfh = agg[agg['head_'] == head]
        plt.errorbar(dfh['sample_size_'], dfh[mean_col],
                     yerr=dfh[std_col], marker='o', capsize=4, label=head.upper())
    plt.title(f"{label} vs. Sample Size (Test, Best Epoch)")
    plt.xlabel("Number of Training Samples")
    plt.ylabel(label)
    plt.grid(True)
    plt.legend(title="Head Type")
    plt.tight_layout()
    #plt.savefig(f"figures/summary/{label.lower().replace('²','2')}_vs_samplesize.png")
    plt.show()


In [ ]:
# ------------------------------------------------------------------------------
# 5. Distribution comparison on full dataset
# ------------------------------------------------------------------------------
# Re-train full models to get predictions
preds = {}
for head in ["regression", "mdn"]:
    model = FTTransformerModel(
        sample_df=X_train,
        seed=42,
        use_mdn=(head == "mdn"),
        **common_config
    )
    model.train(X_train[all_cols], y_train, X_test[all_cols], y_test)
    preds[head] = model.predict(X_test)

y_true = y_test

# Scatter plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, head in zip(axes, ["regression", "mdn"]):
    ax.scatter(y_true, preds[head], alpha=0.5)
    mn, mx = np.min(y_true), np.max(y_true)
    ax.plot([mn, mx], [mn, mx])
    ax.set_title(f'{head.capitalize()} Head: True vs Pred')
    ax.set_xlabel('True')
    ax.set_ylabel('Pred')
    ax.grid(True)
plt.tight_layout()
#plt.savefig("figures/scatter_true_vs_pred.png")
plt.show()

# Histograms of distributions
fig, axes = plt.subplots(2, 1, figsize=(8, 10))
for ax, head in zip(axes, ["regression", "mdn"]):
    ax.hist(y_true, bins=40, density=True, alpha=0.5, label='True')
    ax.hist(preds[head], bins=40, density=True, alpha=0.5, label='Pred')
    ax.set_title(f'{head.capitalize()} Head: Prediction vs True Distribution')
    ax.legend()
    ax.grid(True)
plt.tight_layout()
#plt.savefig("figures/distribution_comparison.png")
plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import up5_helper
from models.ft_transformer_mdn_transfer5_RegressionHeadAblation import FTTransformerModel
from sklearn.metrics import r2_score

# ------------------------------------------------------------------------------
# 1. Load and prepare dataset
# ------------------------------------------------------------------------------
X_train, X_test, y_train, y_test = up5_helper.load_dataset("../data/UP5/", only_36=False)
X_train = up5_helper.reconstruct_categories(X_train)
X_test = up5_helper.reconstruct_categories(X_test)
all_cols = list(X_train.columns)

# ------------------------------------------------------------------------------
# 2. Common training configuration
# ------------------------------------------------------------------------------
common_config = {
    'epochs': 80,
    'deterministic': True,
    'benchmark': False,
    'verbose': False,
    'early_stopping': False,
    'd_token': 56,
    'n_heads': 4,
    'n_blocks': 4,
    'ff_factor': 8.0,
    'dropout': 0.15,
    'learning_rate': 3e-4,
    'weight_decay': 5e-5,
    'mdn_n_components': 2,
    'mdn_temperature': 2.5,
    'mdn_hidden_size': 24,
    'batch_size': 32,
    'use_interaction': True,
    'use_scheduler': False
}

In [ ]:
# ------------------------------------------------------------------------------
# 3. Experiment: Vary sample size and head type
# ------------------------------------------------------------------------------
sample_sizes = [100, 500, 1000, len(X_train)]
results = []

rng = np.random.RandomState(42)
for head in ["regression", "mdn"]:
    for n in sample_sizes:
        print(f"Running: head={head}, sample_size={n}")
        idx = rng.choice(len(X_train), size=n, replace=False)
        X_sub = X_train.iloc[idx]
        y_sub = y_train[idx]

        model = FTTransformerModel(
            sample_df=X_sub,
            seed=42,
            use_mdn=(head == "mdn"),
            **common_config
        )
        model.train(X_sub[all_cols], y_sub, X_test[all_cols], y_test)
        hist = model.get_training_history()

        # Record metrics at best epoch
        r2_best = hist['val_r2'][model.best_epoch]
        jsd_best = hist['val_jsd'][model.best_epoch]
        rmse_best = hist['val_rmse'][model.best_epoch]
        
        # Record metrics at final epoch
        r2_final = hist['val_r2'][-1]
        jsd_final = hist['val_jsd'][-1]
        rmse_final = hist['val_rmse'][-1]

        results.append({
            'head_type': head,
            'sample_size': n,
            'r2 final': r2_final,
            'r2 best': r2_best,
            'jsd final': jsd_final,
            'jsd best': jsd_best,
            'rmse final': rmse_final,
            'rmse best': rmse_best
        })

# Aggregate results into DataFrame
df_results = pd.DataFrame(results)
os.makedirs("figures", exist_ok=True)

# Display table
print("Summary of metrics:")
print(df_results.to_markdown(index=False))


In [ ]:



# ------------------------------------------------------------------------------
# 4. Line plots: Metric vs Sample Size
# ------------------------------------------------------------------------------
metrics = ['r2', 'jsd', 'rmse']
for metric in metrics:
    plt.figure(figsize=(6, 4))
    for head in df_results.head_type.unique():
        subset = df_results[df_results.head_type == head]
        plt.plot(subset.sample_size, subset[metric], marker='o', label=head)
    plt.xlabel('Sample Size')
    plt.ylabel(metric.upper())
    plt.title(f'{metric.upper()} vs. Sample Size')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    #plt.savefig(f"figures/{metric}_vs_sample_size.png")
    plt.show()

# ------------------------------------------------------------------------------
# 5. Distribution comparison on full dataset
# ------------------------------------------------------------------------------
# Re-train full models to get predictions
preds = {}
for head in ["regression", "mdn"]:
    model = FTTransformerModel(
        sample_df=X_train,
        seed=42,
        use_mdn=(head == "mdn"),
        **common_config
    )
    model.train(X_train[all_cols], y_train, X_test[all_cols], y_test)
    preds[head] = model.predict(X_test)

y_true = y_test

# Scatter plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, head in zip(axes, ["regression", "mdn"]):
    ax.scatter(y_true, preds[head], alpha=0.5)
    mn, mx = np.min(y_true), np.max(y_true)
    ax.plot([mn, mx], [mn, mx])
    ax.set_title(f'{head.capitalize()} Head: True vs Pred')
    ax.set_xlabel('True')
    ax.set_ylabel('Pred')
    ax.grid(True)
plt.tight_layout()
#plt.savefig("figures/scatter_true_vs_pred.png")
plt.show()

# Histograms of distributions
fig, axes = plt.subplots(2, 1, figsize=(8, 10))
for ax, head in zip(axes, ["regression", "mdn"]):
    ax.hist(y_true, bins=40, density=True, alpha=0.5, label='True')
    ax.hist(preds[head], bins=40, density=True, alpha=0.5, label='Pred')
    ax.set_title(f'{head.capitalize()} Head: Prediction vs True Distribution')
    ax.legend()
    ax.grid(True)
plt.tight_layout()
#plt.savefig("figures/distribution_comparison.png")
plt.show()

# ------------------------------------------------------------------------------
# 6. Save results for inclusion
# ------------------------------------------------------------------------------
#df_results.to_csv("figures/ablation_results.csv", index=False)
#print("All figures and CSV have been saved in the 'figures/' directory.")


### Transfer

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import up5_helper
from models.ft_transformer_mdn_transfer5_RegressionHeadAblation import FTTransformerModel

# Load & reconstruct dataset
X_train, X_test, y_train, y_test = up5_helper.load_dataset("../data/UP5/", only_36=False)
X_train = up5_helper.reconstruct_categories(X_train)
X_test = up5_helper.reconstruct_categories(X_test)

shared_columns = [c[:-1] if c[-1] == "_" else c for c in up5_helper.get_36_columns()]
all_cols = list(X_train.columns)

In [ ]:
# Base training config
common_config = {
    'epochs': 80,
    'deterministic': True,
    'benchmark': False,
    'verbose': False,
    'early_stopping': False,
    'd_token': 56,
    'n_heads': 4,
    'n_blocks': 4,
    'ff_factor': 8.0,
    'dropout': 0.15,
    'learning_rate': 3e-4,
    'weight_decay': 5e-5,
    'mdn_n_components': 2,
    'mdn_temperature': 2.5,
    'mdn_hidden_size': 24,
    'batch_size': 32,
    'use_interaction': True,
    'use_scheduler': False
}

# Results container
results = []


In [ ]:
# Loop over MDN and Regression
for method in ["mdn","regression"]:
    print(f"\n=== Running ablation: {method.upper()} ===")
    use_mdn = method == "mdn"

    # TARGET ONLY
    model = FTTransformerModel(
        sample_df=X_train[all_cols],
        seed=42,
        use_mdn=use_mdn,
        **common_config
    )
    model.train(X_train[all_cols], y_train, X_test[all_cols], y_test)

    results.append({
        "method": f"{method}_target_only",
        "val_r2": model.get_training_history()["val_r2"]
    })

    # TRANSFER LEARNING
    model_transfer = FTTransformerModel(
        sample_df=X_train[shared_columns],
        seed=42,
        use_mdn=use_mdn,
        **common_config
    )
    print("  Pre-training on source (shared features)…")
    model_transfer.train(X_train[shared_columns], y_train, X_test[shared_columns], y_test)

    print("  Fine-tuning on target (all features)…")
    model_transfer.patience = 20
    model_transfer.fine_tune(
        X_new_train=X_train[all_cols],
        y_new_train=y_train,
        X_new_val=X_test[all_cols],
        y_new_val=y_test,
        additional_epochs=80,
        freeze_backbone=False,
        freeze_shared=False
    )

    val_r2_all = model_transfer.get_training_history()["val_r2"]
    max_train_epoch = model_transfer.max_train_epoch
    max_finetune_epoch = model_transfer.max_finetune_epoch

    results.append({
        "method": f"{method}_transfer_pretrain",
        "val_r2": val_r2_all[:max_train_epoch]
    })

    results.append({
        "method": f"{method}_transfer_finetune",
        "val_r2": val_r2_all[max_train_epoch:max_train_epoch + max_finetune_epoch]
    })

# Flatten results
records = []
for r in results:
    for epoch, r2 in enumerate(r["val_r2"], 1):
        records.append({
            "method": r["method"],
            "epoch": epoch,
            "r2": r2
        })

df_ablation = pd.DataFrame(records)
print("✅ Ablation complete. DataFrame created.")

# Save or visualize if needed:
# df_ablation.to_csv("ablation_mdn_vs_regression.csv", index=False)


In [ ]:
df_ablation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Step 1: Phase visibility flags
SHOW_TARGET_ONLY = True
SHOW_TRANSFER_PRETRAIN = True
SHOW_TRANSFER_FINETUNE = True

# Step 2: Extract method components
df_ablation["head_type"] = df_ablation["method"].str.extract(r"^(mdn|regression)")
df_ablation["phase"] = df_ablation["method"].str.extract(r"_(target_only|transfer_pretrain|transfer_finetune)$")

# Step 3: Filter phases based on the booleans
visible_phases = []
if SHOW_TARGET_ONLY:
    visible_phases.append("target_only")
if SHOW_TRANSFER_PRETRAIN:
    visible_phases.append("transfer_pretrain")
if SHOW_TRANSFER_FINETUNE:
    visible_phases.append("transfer_finetune")

filtered_df = df_ablation[df_ablation["phase"].isin(visible_phases)]

# Step 4: Custom colors
custom_palette = {
    "mdn": "#EF7C00",         # Orange
    "regression": "#003D7C"   # Navy
}

# Step 5: Plot
plt.figure(figsize=(10, 5))
sns.set(style="whitegrid")

sns.lineplot(
    data=filtered_df,
    x="epoch", y="r2",
    hue="head_type",        
    style="phase",          
    style_order=["transfer_finetune", "target_only", "transfer_pretrain"],
    palette=custom_palette,
    linewidth=2.2
)

plt.title("R² Comparison: MDN vs Regression Head")
plt.ylabel("Validation R²")
plt.xlabel("Epoch")
plt.legend(title="Head Type / Phase", loc="best")
plt.tight_layout()
plt.show()


In [ ]:
# Loop over MDN and Regression
for method in ["mdn","regression"]:
    print(f"\n=== Running ablation: {method.upper()} ===")
    use_mdn = method == "mdn"

    # TARGET ONLY
    model = FTTransformerModel(
        sample_df=X_train[all_cols],
        seed=42,
        use_mdn=use_mdn,
        **common_config
    )
    model.train(X_train[all_cols], y_train, X_test[all_cols], y_test)

    results.append({
        "method": f"{method}_target_only",
        "val_r2": model.get_training_history()["val_r2"]
    })

    # TRANSFER LEARNING
    model_transfer = FTTransformerModel(
        sample_df=X_train[shared_columns],
        seed=42,
        use_mdn=use_mdn,
        **common_config
    )
    print("  Pre-training on source (shared features)…")
    model_transfer.train(X_train[shared_columns], y_train, X_test[shared_columns], y_test)

    print("  Fine-tuning on target (all features)…")
    model_transfer.patience = 20
    model_transfer.fine_tune(
        X_new_train=X_train[all_cols],
        y_new_train=y_train,
        X_new_val=X_test[all_cols],
        y_new_val=y_test,
        additional_epochs=80,
        freeze_backbone=False,
        freeze_shared=False
    )

    val_r2_all = model_transfer.get_training_history()["val_r2"]
    max_train_epoch = model_transfer.max_train_epoch
    max_finetune_epoch = model_transfer.max_finetune_epoch

    results.append({
        "method": f"{method}_transfer_pretrain",
        "val_r2": val_r2_all[:max_train_epoch]
    })

    results.append({
        "method": f"{method}_transfer_finetune",
        "val_r2": val_r2_all[max_train_epoch:max_train_epoch + max_finetune_epoch]
    })

# Flatten results
records = []
for r in results:
    for epoch, r2 in enumerate(r["val_r2"], 1):
        records.append({
            "method": r["method"],
            "epoch": epoch,
            "r2": r2
        })

df_ablation = pd.DataFrame(records)
print("✅ Ablation complete. DataFrame created.")

# Save or visualize if needed:
# df_ablation.to_csv("ablation_mdn_vs_regression.csv", index=False)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Step 1: Phase visibility flags
SHOW_TARGET_ONLY = True
SHOW_TRANSFER_PRETRAIN = True
SHOW_TRANSFER_FINETUNE = True

# Step 2: Extract method components
df_ablation["head_type"] = df_ablation["method"].str.extract(r"^(mdn|regression)")
df_ablation["phase"] = df_ablation["method"].str.extract(r"_(target_only|transfer_pretrain|transfer_finetune)$")

# Step 3: Filter phases based on the booleans
visible_phases = []
if SHOW_TARGET_ONLY:
    visible_phases.append("target_only")
if SHOW_TRANSFER_PRETRAIN:
    visible_phases.append("transfer_pretrain")
if SHOW_TRANSFER_FINETUNE:
    visible_phases.append("transfer_finetune")

filtered_df = df_ablation[df_ablation["phase"].isin(visible_phases)]

# Step 4: Custom colors
custom_palette = {
    "mdn": "#EF7C00",         # Orange
    "regression": "#003D7C"   # Navy
}

# Step 5: Plot
plt.figure(figsize=(10, 5))
sns.set(style="whitegrid")

sns.lineplot(
    data=filtered_df,
    x="epoch", y="r2",
    hue="head_type",        
    style="phase",          
    style_order=["transfer_finetune", "target_only", "transfer_pretrain"],
    palette=custom_palette,
    linewidth=2.2
)

plt.title("R² Comparison: MDN vs Regression Head")
plt.ylabel("Validation R²")
plt.xlabel("Epoch")
plt.legend(title="Head Type / Phase", loc="best")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Step 1: Phase visibility flags
SHOW_TARGET_ONLY = True
SHOW_TRANSFER_PRETRAIN = True
SHOW_TRANSFER_FINETUNE = True

# Step 2: Extract method components
df_ablation["head_type"] = df_ablation["method"].str.extract(r"^(mdn|regression)")
df_ablation["phase"] = df_ablation["method"].str.extract(r"_(target_only|transfer_pretrain|transfer_finetune)$")

# Step 3: Filter phases based on the booleans
visible_phases = []
if SHOW_TARGET_ONLY:
    visible_phases.append("target_only")
if SHOW_TRANSFER_PRETRAIN:
    visible_phases.append("transfer_pretrain")
if SHOW_TRANSFER_FINETUNE:
    visible_phases.append("transfer_finetune")

filtered_df = df_ablation[df_ablation["phase"].isin(visible_phases)]

# Step 4: Custom colors
custom_palette = {
    "mdn": "#EF7C00",         # Orange
    "regression": "#003D7C"   # Navy
}

# Step 5: Plot
plt.figure(figsize=(10, 5))
sns.set(style="whitegrid")

sns.lineplot(
    data=filtered_df,
    x="epoch", y="r2",
    hue="head_type",        
    style="phase",          
    style_order=["transfer_finetune", "target_only", "transfer_pretrain"],
    palette=custom_palette,
    linewidth=2.2
)

plt.title("R² Comparison: MDN vs Regression Head")
plt.ylabel("Validation R²")
plt.xlabel("Epoch")
plt.legend(title="Head Type / Phase", loc="best")
plt.tight_layout()
plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product

import up5_helper
from models.ft_transformer_mdn_transfer4_RegressionHeadAblation import FTTransformerModel

In [ ]:
# Load data
X_train, X_test, y_train, y_test = up5_helper.load_dataset("../data/UP5/", only_36=False)
X_train_reconstructed = up5_helper.reconstruct_categories(X_train)
X_test_reconstructed = up5_helper.reconstruct_categories(X_test)

In [ ]:
# Parameters
shared_columns = [c[:-1] if c[-1] == "_" else c for c in up5_helper.get_36_columns()]
all_cols = list(X_train_reconstructed.columns)
sample_fractions = [0.25, 0.5, 0.75, 1.0]
n_repeats = 15  # to reduce runtime in this test setup

In [ ]:
best_param = {
    'd_token': 56,             # higher dimensionality performed best
    'n_heads': 4,              # still compatible (56 / 4 = 14)
    'n_blocks': 4,             # deeper transformer
    'ff_factor': 8.0,          # stable and optimal
    'dropout': 0.15,           # consistent performer
    'learning_rate': 3e-4,     # lower learning rate worked best
    'weight_decay': 5e-5,      # light regularization helped
    'mdn_n_components': 2,     # unchanged
    'mdn_temperature': 2.5,    # slightly lower temperature worked best
    'mdn_hidden_size': 24,     # best run used smaller MDN
    'batch_size': 32,          # best config used this
    'use_interaction': True,   # enabled in best config
    'use_scheduler': False     # scheduler off in best config
}

additional_config = {
    'epochs': 80,             # still needed for late peaking
    #'seed': 42,                # reproducibility
    'deterministic': True,     # keep results consistent
    'benchmark': False,        # disables cudnn autotune for determinism
    'verbose': False,           # show training logs
    'early_stopping': True
}

In [ ]:
results = []

# Run training for each fraction and repetition
for frac, run in product(sample_fractions, range(n_repeats)):
    print(f"\n>>> Fraction {frac:.2f}, run {run} – sampling data")
    n_samples = int(len(X_train_reconstructed) * frac)
    X_sample = X_train_reconstructed.sample(n=n_samples, random_state=run)
    y_sample = y_train[X_sample.index]

    # -------------------- TARGET ONLY --------------------
    model_tgt = FTTransformerModel(
        sample_df=X_sample[all_cols],
        seed=run,
        **additional_config,
        **best_param
    )
    print("  [target_only] training …")
    model_tgt.train(
        X_sample[all_cols],
        y_sample,
        X_test_reconstructed[all_cols],
        y_test
    )

    results.append({
        "fraction": frac,
        "method": "target_only",
        "run": run,
        "val_r2": model_tgt.get_training_history()["val_r2"]
    })

    # -------------------- TRANSFER LEARNING --------------------
    model_transfer = FTTransformerModel(
        sample_df=X_train_reconstructed[shared_columns],
        seed=run,
        **additional_config,
        **best_param
    )
    print("  [transfer] pre-training on source …")
    model_transfer.train(
        X_train_reconstructed[shared_columns],
        y_train,
        X_test_reconstructed[shared_columns],
        y_test
    )
    print("  [transfer] fine-tuning on target …")
    model_transfer.patience = 20
    model_transfer.fine_tune(      
        X_new_train=X_sample[all_cols],
        y_new_train=y_sample,
        X_new_val=X_test_reconstructed[all_cols],
        y_new_val=y_test,
        additional_epochs=80,
        freeze_backbone=False,
        freeze_shared=False
    )


    # Split val_r2 into pretrain and finetune
    val_r2_all = model_transfer.get_training_history()["val_r2"]
    max_train_epoch = model_transfer.max_train_epoch
    max_finetune_epoch = model_transfer.max_finetune_epoch

    val_r2_pre = val_r2_all[:max_train_epoch]
    val_r2_fine = val_r2_all[max_train_epoch:max_train_epoch + max_finetune_epoch]

    # Save pretrain part
    results.append({
        "fraction": frac,
        "method": "transfer_pretrain",
        "run": run,
        "val_r2": val_r2_pre
    })

    # Save finetune part
    results.append({
        "fraction": frac,
        "method": "transfer_finetune",
        "run": run,
        "val_r2": val_r2_fine
    })

    
records = []
for r in results:
    for epoch, r2 in enumerate(r["val_r2"], 1):
        records.append({
            "fraction": r["fraction"],
            "method": r["method"],
            "run": r["run"],
            "epoch": epoch,
            "r2": r2
        })

df_all = pd.DataFrame(records)
print("All done! Summary dataframe created.")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["font.family"] = "Times New Roman"

# ------------------------- Configuration -------------------------
max_ep = 80
min_runs = 2
show_pretrain = False

# Optional mapping for human-readable method names
method_label_map = {
    "target_only": "Target Only",
    "transfer_pretrain": "Transfer (Pretrain)",
    "transfer_finetune": "Transfer (Finetune)"
}

# ------------------------- Forward Fill -------------------------
def forward_fill(group):
    r = group.set_index("epoch").reindex(range(1, max_ep + 1))
    r["r2"] = r["r2"].ffill()
    r["run"] = group["run"].iloc[0]
    r["fraction"] = group["fraction"].iloc[0]
    r["method"] = group["method"].iloc[0]
    return r.reset_index()

df_filled = (
    df_all.groupby(["fraction", "method", "run"])
    .apply(forward_fill)
    .reset_index(drop=True)
)

# Apply method name mapping
df_all_tmp = df_filled.copy()
df_all_tmp["method"] = df_all_tmp["method"].map(method_label_map)

# ------------------------- Filter by Valid Epochs -------------------------
valid_epochs = (
    df_all_tmp
    .groupby(["fraction", "method", "epoch"])
    .size()
    .reset_index(name="count")
    .query("count >= @min_runs")
    .drop(columns="count")
)

df_all_tmp = pd.merge(df_all_tmp, valid_epochs, on=["fraction", "method", "epoch"])

# ------------------------- Summary Statistics -------------------------
def summarize_group(df):
    return df.groupby(["fraction", "method", "epoch"])["r2"].agg(
        mean="mean",
        q25=lambda x: np.percentile(x, 25),
        q75=lambda x: np.percentile(x, 75)
    ).reset_index()

summary_df = summarize_group(df_all_tmp)

# ------------------------- Dropout Marker Calculation -------------------------
# Last real epoch (before forward filling)
last_real_epochs = (
    df_all.groupby(["fraction", "method", "run"])["epoch"]
    .max()
    .reset_index()
    .rename(columns={"epoch": "last_real_epoch"})
)

# Merge with filled data
df_markers = pd.merge(df_filled, last_real_epochs, on=["fraction", "method", "run"])
marker_df = df_markers[df_markers["epoch"] == df_markers["last_real_epoch"]]

# Compute mean R² at dropout points
marker_summary = (
    marker_df.groupby(["fraction", "method", "epoch"])["r2"]
    .mean()
    .reset_index()
    .rename(columns={"r2": "r2_marker"})
)
marker_summary["method"] = marker_summary["method"].map(method_label_map)

# ------------------------- Color Setup -------------------------
fractions_sorted = sorted(df_all["fraction"].unique())
palette = sns.color_palette("tab10", n_colors=len(fractions_sorted))
color_map = {frac: palette[i] for i, frac in enumerate(fractions_sorted)}

# ------------------------- Plot -------------------------
plt.figure(figsize=(12, 6))

for frac in fractions_sorted:
    df_frac = summary_df[summary_df["fraction"] == frac]
    color = color_map[frac]

    method_styles = [
        ("Transfer (Finetune)", "-"),
        ("Target Only", "--")
    ]
    if show_pretrain:
        method_styles.insert(0, ("Transfer (Pretrain)", "--"))

    for method, linestyle in method_styles:
        df_m = df_frac[df_frac["method"] == method]
        if df_m.empty:
            continue

        label = f"{int(frac * 100)}% - {method}"
        plt.plot(df_m["epoch"], df_m["mean"], linestyle=linestyle,
                 label=label, color=color, linewidth=2)
        plt.fill_between(df_m["epoch"], df_m["q25"], df_m["q75"],
                         alpha=0.05, color=color)

        # Plot marker where real training stopped
        if method == "Transfer (Finetune)":
            df_marker = marker_summary[
                (marker_summary["fraction"] == frac) & 
                (marker_summary["method"] == method)
            ]
            if not df_marker.empty:
                plt.scatter(df_marker["epoch"], df_marker["r2_marker"],
                            marker="^", s=20, color=color, zorder=5)


# ------------------------- Final Touches -------------------------
plt.title("UP5 Data: Heterogeneous Feature Transfer Learning vs Target-Only by Train Fraction", fontsize=15)
plt.xlabel("Epoch", fontsize=13)
plt.ylabel("Validation R²", fontsize=13)
plt.ylim(-0.05, 0.65)
plt.grid(True)

# Custom legend ordering
def legend_sort_key(label):
    frac_str, method = label.split(" - ")
    frac = int(frac_str.replace("%", ""))
    priority = {"Transfer (Finetune)": 0, "Target Only": 1, "Transfer (Pretrain)": 2}
    return (-frac, priority.get(method, 99))

handles, labels = plt.gca().get_legend_handles_labels()
sorted_items = sorted(zip(handles, labels), key=lambda x: legend_sort_key(x[1]))
handles_sorted, labels_sorted = zip(*sorted_items)
plt.legend(handles_sorted, labels_sorted, fontsize=10, loc="lower right")

plt.tight_layout()

plt.savefig(f"ft_mdn_transformer_transfer_up5_only_heterogeneous_differentDS_{up5_helper.timestamp()}.pdf")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["font.family"] = "Times New Roman"

# ------------------------- Configuration -------------------------
max_ep = 80
min_runs = 2
show_pretrain = False

# Optional mapping for human-readable method names
method_label_map = {
    "target_only": "Target Only",
    "transfer_pretrain": "Transfer (Pretrain)",
    "transfer_finetune": "Transfer (Finetune)"
}

# ------------------------- Forward Fill -------------------------
def forward_fill(group):
    r = group.set_index("epoch").reindex(range(1, max_ep + 1))
    r["r2"] = r["r2"].ffill()
    r["run"] = group["run"].iloc[0]
    r["fraction"] = group["fraction"].iloc[0]
    r["method"] = group["method"].iloc[0]
    return r.reset_index()

df_filled = (
    df_all.groupby(["fraction", "method", "run"])
    .apply(forward_fill)
    .reset_index(drop=True)
)

# Apply method name mapping
df_all_tmp = df_filled.copy()
df_all_tmp["method"] = df_all_tmp["method"].map(method_label_map)

# ------------------------- Filter by Valid Epochs -------------------------
valid_epochs = (
    df_all_tmp
    .groupby(["fraction", "method", "epoch"])
    .size()
    .reset_index(name="count")
    .query("count >= @min_runs")
    .drop(columns="count")
)

df_all_tmp = pd.merge(df_all_tmp, valid_epochs, on=["fraction", "method", "epoch"])

# ------------------------- Summary Statistics -------------------------
def summarize_group(df):
    return df.groupby(["fraction", "method", "epoch"])["r2"].agg(
        mean="mean",
        q25=lambda x: np.percentile(x, 25),
        q75=lambda x: np.percentile(x, 75)
    ).reset_index()

summary_df = summarize_group(df_all_tmp)

# ------------------------- Compute Method Differences -------------------------
pivoted = summary_df.pivot_table(index=["fraction", "epoch"], 
                                 columns="method", values="mean").reset_index()

# Only keep rows where Transfer (Finetune) outperforms Target Only
pivoted = pivoted.dropna(subset=["Transfer (Finetune)", "Target Only"])
pivoted = pivoted[pivoted["Transfer (Finetune)"] > pivoted["Target Only"]]

pivoted["r2_diff_top"] = pivoted["Transfer (Finetune)"]
pivoted["r2_diff_bottom"] = pivoted["Target Only"]


# ------------------------- Dropout Marker Calculation -------------------------
# Last real epoch (before forward filling)
last_real_epochs = (
    df_all.groupby(["fraction", "method", "run"])["epoch"]
    .max()
    .reset_index()
    .rename(columns={"epoch": "last_real_epoch"})
)

# Merge with filled data
df_markers = pd.merge(df_filled, last_real_epochs, on=["fraction", "method", "run"])
marker_df = df_markers[df_markers["epoch"] == df_markers["last_real_epoch"]]

# Compute mean R² at dropout points
marker_summary = (
    marker_df.groupby(["fraction", "method", "epoch"])["r2"]
    .mean()
    .reset_index()
    .rename(columns={"r2": "r2_marker"})
)
marker_summary["method"] = marker_summary["method"].map(method_label_map)

# ------------------------- Color Setup -------------------------
fractions_sorted = sorted(df_all["fraction"].unique())
palette = sns.color_palette("tab10", n_colors=len(fractions_sorted))
color_map = {frac: palette[i] for i, frac in enumerate(fractions_sorted)}

# ------------------------- Plot -------------------------
plt.figure(figsize=(12, 6))

for frac in fractions_sorted:
    df_frac = summary_df[summary_df["fraction"] == frac]
    color = color_map[frac]

    method_styles = [
        ("Transfer (Finetune)", "-"),
        ("Target Only", "--")
    ]
    if show_pretrain:
        method_styles.insert(0, ("Transfer (Pretrain)", "--"))

    for method, linestyle in method_styles:
        df_m = df_frac[df_frac["method"] == method]
        if df_m.empty:
            continue

        label = f"{int(frac * 100)}% - {method}"
        plt.plot(df_m["epoch"], df_m["mean"], linestyle=linestyle,
                 label=label, color=color, linewidth=1.8)
        # Fill between transfer and target
        df_diff = pivoted[pivoted["fraction"] == frac]
        plt.fill_between(df_diff["epoch"], df_diff["r2_diff_bottom"], df_diff["r2_diff_top"],
                 alpha=0.1, color=color)



        # Plot marker where real training stopped
        if method == "Transfer (Finetune)":
            df_marker = marker_summary[
                (marker_summary["fraction"] == frac) & 
                (marker_summary["method"] == method)
            ]
            if not df_marker.empty:
                plt.scatter(df_marker["epoch"], df_marker["r2_marker"],
                            marker="^", s=20, color=color, zorder=5)


# ------------------------- Final Touches -------------------------
plt.title("UP5 Data: Heterogeneous Feature Transfer Learning vs Target-Only by Train Fraction", fontsize=15)
plt.xlabel("Epoch", fontsize=13)
plt.ylabel("Validation R²", fontsize=13)
plt.ylim(-0.05, 0.65)
plt.grid(True)

# Custom legend ordering
def legend_sort_key(label):
    frac_str, method = label.split(" - ")
    frac = int(frac_str.replace("%", ""))
    priority = {"Transfer (Finetune)": 0, "Target Only": 1, "Transfer (Pretrain)": 2}
    return (-frac, priority.get(method, 99))

handles, labels = plt.gca().get_legend_handles_labels()
sorted_items = sorted(zip(handles, labels), key=lambda x: legend_sort_key(x[1]))
handles_sorted, labels_sorted = zip(*sorted_items)
plt.legend(handles_sorted, labels_sorted, fontsize=10, loc="lower right")

plt.tight_layout()

plt.savefig(f"ft_mdn_transformer_transfer_up5_only_heterogeneous_differentDS_{up5_helper.timestamp()}.pdf")
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# ------------------------- Compute Best Scores -------------------------
best_scores = (
    df_all.groupby(["fraction", "method", "run"])["r2"]
    .max()
    .reset_index()
)

# ------------------------- Map and Filter Methods -------------------------
method_label_map = {
    "target_only": "Target Only",
    "transfer_finetune": "Transfer (Finetune)"
}
best_scores["method"] = best_scores["method"].map(method_label_map)

methods_to_show = ["Target Only", "Transfer (Finetune)"]
best_scores = best_scores[best_scores["method"].isin(methods_to_show)]

# ------------------------- Custom Color Palette -------------------------
custom_palette = {
    "Target Only": "#5b89b0",          # blue
    "Transfer (Finetune)": "#EF7C00"   # orange
}

# ------------------------- Plot -------------------------
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=best_scores,
    x="fraction",
    y="r2",
    hue="method",
    palette=custom_palette,
    linewidth=1.2,
    fliersize=2
)

# ------------------------- Styling -------------------------
plt.title("UP5 Data: Heterogeneous Features Best Validation R² Avg per 15 Runs", fontsize=14)
plt.xlabel("Training Fraction", fontsize=12)
plt.ylabel("Best R²", fontsize=12)
plt.ylim(-0.05, 0.65)
plt.legend(title="", loc="lower right", fontsize=10)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()

# Optional save
plt.savefig(f"up5_hetero_best_r2_transfer_vs_target_only{up5_helper.timestamp()}.pdf")

plt.show()


In [ ]:
max_ep = 80                                        # or df_all["epoch"].max()
def forward_fill(group):
    r = group.set_index("epoch").reindex(range(1, max_ep+1))
    r["r2"] = r["r2"].ffill()                      # last-observation-carried-forward
    r["run"] = group["run"].iloc[0]                # keep id columns
    r["fraction"] = group["fraction"].iloc[0]
    r["method"]   = group["method"].iloc[0]
    return r.reset_index()

df_filled = (df_all.groupby(["fraction","method","run"])
                    .apply(forward_fill)
                    .reset_index(drop=True))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.rcParams["font.family"] = "Times New Roman"

# Toggle to optionally show the pretraining phase in the plot
show_pretrain = False  

# Prepare color mapping for each training data fraction
fractions_sorted = sorted(df_all["fraction"].unique())
palette = sns.color_palette("tab10", n_colors=len(fractions_sorted))
color_map = {frac: palette[i] for i, frac in enumerate(fractions_sorted)}

# Human-readable method labels
method_label_map = {
    "target_only": "Target Only",
    "transfer_pretrain": "Transfer (Pretrain)",
    "transfer_finetune": "Transfer (Finetune)"
}
df_all_tmp = df_filled.copy()
df_all_tmp["method"] = df_all_tmp["method"].map(method_label_map)

# Group and summarize R² values: mean and interquartile range
def summarize_group(df):
    return df.groupby(["fraction", "method", "epoch"])["r2"].agg(
        mean="mean",
        q25=lambda x: np.percentile(x, 25),
        q75=lambda x: np.percentile(x, 75)
    ).reset_index()

# Remove epochs with fewer than x runs per group
min_runs = 4

valid_epochs = (
    df_all_tmp
    .groupby(["fraction", "method", "epoch"])
    .size()
    .reset_index(name="count")
    .query("count >= @min_runs")
    .drop(columns="count")
)

# Merge to filter df_all_tmp to only those epochs
df_all_tmp = pd.merge(df_all_tmp, valid_epochs, on=["fraction", "method", "epoch"])





summary_df = summarize_group(df_all_tmp)

# Initialize plot
plt.figure(figsize=(12, 6))

# Loop through each training data fraction
for frac in fractions_sorted:
    df_frac = summary_df[summary_df["fraction"] == frac]
    color = color_map[frac]

    # Define line style per method
    method_styles = [
        ("Transfer (Finetune)", "-"),  # solid line
        ("Target Only", ":")          # dashed line
    ]
    if show_pretrain:
        method_styles.insert(0, ("Transfer (Pretrain)", ":"))  # dotted line if enabled

    # Plot each method
    for method, linestyle in method_styles:
        df_m = df_frac[df_frac["method"] == method]
        if df_m.empty:
            continue
        label = f"{int(frac * 100)}% - {method}"
        plt.plot(df_m["epoch"], df_m["mean"], linestyle=linestyle, label=label, color=color, linewidth=2)
        plt.fill_between(df_m["epoch"], df_m["q25"], df_m["q75"], alpha=0.05, color=color)

# Final plot styling
plt.title("UP5 Data: Heterogeneous Feature Transfer Learning vs Target-Only by Train Fraction", fontsize=15)
plt.xlabel("Epoch", fontsize=13)
plt.ylabel("Validation R²", fontsize=13)
plt.ylim(-0.05, 0.65)
#plt.xlim(-0.05, 80)
plt.grid(True)
handles, labels = plt.gca().get_legend_handles_labels()
# Custom legend ordering: 100% on top, Transfer before Target
from matplotlib.lines import Line2D

# Grab handles and labels
handles, labels = plt.gca().get_legend_handles_labels()

# Define custom sort key
def legend_sort_key(label):
    parts = label.split(" - ")
    frac = int(parts[0].replace("%", ""))
    method_priority = {
        "Transfer (Finetune)": 0,
        "Target Only": 1,
        "Transfer (Pretrain)": 2  # optional
    }
    method = parts[1]
    return (-frac, method_priority.get(method, 99))  # negative for descending fraction

# Sort handles and labels accordingly
sorted_pairs = sorted(zip(handles, labels), key=lambda x: legend_sort_key(x[1]))
handles_sorted, labels_sorted = zip(*sorted_pairs)

# Apply sorted legend
plt.legend(handles_sorted, labels_sorted, fontsize=10, loc="lower right")


plt.tight_layout()

#plt.savefig(f"ft_mdn_transformer_transfer_up5_only_heterogeneous_differentDS_{up5_helper.timestamp()}.pdf")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.rcParams["font.family"] = "Times New Roman"

# Toggle to optionally show the pretraining phase in the plot
show_pretrain = False  

# Prepare color mapping for each training data fraction
fractions_sorted = sorted(df_all["fraction"].unique())
palette = sns.color_palette("tab10", n_colors=len(fractions_sorted))
color_map = {frac: palette[i] for i, frac in enumerate(fractions_sorted)}

# Human-readable method labels
method_label_map = {
    "target_only": "Target Only",
    "transfer_pretrain": "Transfer (Pretrain)",
    "transfer_finetune": "Transfer (Finetune)"
}
df_all_tmp = df_all.copy()
df_all_tmp["method"] = df_all["method"].map(method_label_map)

# Group and summarize R² values: mean and interquartile range
def summarize_group(df):
    return df.groupby(["fraction", "method", "epoch"])["r2"].agg(
        mean="mean",
        q25=lambda x: np.percentile(x, 25),
        q75=lambda x: np.percentile(x, 75)
    ).reset_index()

# Remove epochs with fewer than x runs per group
min_runs = 2

valid_epochs = (
    df_all_tmp
    .groupby(["fraction", "method", "epoch"])
    .size()
    .reset_index(name="count")
    .query("count >= @min_runs")
    .drop(columns="count")
)

# Merge to filter df_all_tmp to only those epochs
df_all_tmp = pd.merge(df_all_tmp, valid_epochs, on=["fraction", "method", "epoch"])





summary_df = summarize_group(df_all_tmp)

# Initialize plot
plt.figure(figsize=(12, 6))

# Loop through each training data fraction
for frac in fractions_sorted:
    df_frac = summary_df[summary_df["fraction"] == frac]
    color = color_map[frac]

    # Define line style per method
    method_styles = [
        ("Transfer (Finetune)", "-"),  # solid line
        ("Target Only", ":")          # dashed line
    ]
    if show_pretrain:
        method_styles.insert(0, ("Transfer (Pretrain)", ":"))  # dotted line if enabled

    # Plot each method
    for method, linestyle in method_styles:
        df_m = df_frac[df_frac["method"] == method]
        if df_m.empty:
            continue
        label = f"{int(frac * 100)}% - {method}"
        plt.plot(df_m["epoch"], df_m["mean"], linestyle=linestyle, label=label, color=color, linewidth=2)
        plt.fill_between(df_m["epoch"], df_m["q25"], df_m["q75"], alpha=0.05, color=color)

# Final plot styling
plt.title("UP5 Data: Heterogeneous Feature Transfer Learning vs Target-Only by Train Fraction", fontsize=15)
plt.xlabel("Epoch", fontsize=13)
plt.ylabel("Validation R²", fontsize=13)
plt.ylim(-0.05, 0.65)
#plt.xlim(-0.05, 80)
plt.grid(True)
handles, labels = plt.gca().get_legend_handles_labels()
# Custom legend ordering: 100% on top, Transfer before Target
from matplotlib.lines import Line2D

# Grab handles and labels
handles, labels = plt.gca().get_legend_handles_labels()

# Define custom sort key
def legend_sort_key(label):
    parts = label.split(" - ")
    frac = int(parts[0].replace("%", ""))
    method_priority = {
        "Transfer (Finetune)": 0,
        "Target Only": 1,
        "Transfer (Pretrain)": 2  # optional
    }
    method = parts[1]
    return (-frac, method_priority.get(method, 99))  # negative for descending fraction

# Sort handles and labels accordingly
sorted_pairs = sorted(zip(handles, labels), key=lambda x: legend_sort_key(x[1]))
handles_sorted, labels_sorted = zip(*sorted_pairs)

# Apply sorted legend
plt.legend(handles_sorted, labels_sorted, fontsize=10, loc="lower right")


plt.tight_layout()

plt.savefig(f"ft_mdn_transformer_transfer_up5_only_heterogeneous_differentDS_{up5_helper.timestamp()}.pdf")
plt.show()


In [ ]:
summary_table = (
    df_all.groupby(["fraction", "method", "run"])["r2"]
    .max()
    .reset_index()
    .groupby(["fraction", "method"])["r2"]
    .agg(mean_r2="mean", std_r2="std")
    .reset_index()
    .sort_values(by=["fraction", "method"])
)

# Format for presentation
summary_table["fraction"] = (summary_table["fraction"] * 100).astype(int).astype(str) + "%"
summary_table["mean_r2"] = summary_table["mean_r2"].round(3)
summary_table["std_r2"] = summary_table["std_r2"].round(3)

# Pivot table for cleaner comparison
pivot_table = summary_table.pivot(index="fraction", columns="method", values=["mean_r2", "std_r2"])
pivot_table.columns = [f"{stat} ({method})" for stat, method in pivot_table.columns]
pivot_table = pivot_table.reset_index()

pivot_table

In [ ]:
5

In [ ]:
results = []

# Run training for each fraction and repetition
for frac, run in product(sample_fractions, range(n_repeats)):
    print(f"\n>>> Fraction {frac:.2f}, run {run} – sampling data")
    n_samples = int(len(X_train_reconstructed) * frac)
    X_sample = X_train_reconstructed.sample(n=n_samples, random_state=run)
    y_sample = y_train[X_sample.index]

    # -------------------- TARGET ONLY --------------------
    model_tgt = FTTransformerModel(
        sample_df=X_sample,
        seed=run,
        **additional_config,
        **best_param
    )
    print("  [target_only] training …")
    model_tgt.train(X_sample, y_sample, X_test_reconstructed, y_test)

    results.append({
        "fraction": frac,
        "method": "target_only",
        "run": run,
        "val_r2": model_tgt.get_training_history()["val_r2"]
    })

    # -------------------- TRANSFER LEARNING --------------------
    model_transfer = FTTransformerModel(
        sample_df=X_train_reconstructed[shared_columns],
        seed=run,
        **additional_config,
        **best_param
    )
    print("  [transfer] pre-training on source …")
    model_transfer.train(
        X_train_reconstructed[shared_columns],
        y_train,
        X_test_reconstructed[shared_columns],
        y_test
    )
    print("  [transfer] fine-tuning on target …")
    model_transfer.fine_tune(      
        X_new_train=X_sample[all_cols],
        y_new_train=y_sample,
        X_new_val=X_test_reconstructed[all_cols],
        y_new_val=y_test,
        additional_epochs=4,
        freeze_backbone=False,
        freeze_shared=False
    )

    results.append({
        "fraction": frac,
        "method": "transfer",
        "run": run,
        "val_r2": model_transfer.get_training_history()["val_r2"]
    })

records = []
for r in results:
    for epoch, r2 in enumerate(r["val_r2"], 1):
        records.append({
            "fraction": r["fraction"],
            "method": r["method"],
            "run": r["run"],
            "epoch": epoch,
            "r2": r2
        })

df_all = pd.DataFrame(records)
print("All done! Summary dataframe created.")


In [ ]:
df_all

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

# Farben für jede Fraction
palette = sns.color_palette("tab10", n_colors=len(sample_fractions))
color_map = {frac: palette[i] for i, frac in enumerate(sample_fractions)}

# Daten extrahieren
def extract_finetune_curves(results):
    records = []
    for r in results:
        method = r["method"]
        val_r2 = r["val_r2"]
        run = r["run"]
        frac = r["fraction"]

        if method == "transfer":
            start = r["max_train_epoch"]
            end = start + r["max_finetune_epoch"]
            val_r2 = val_r2[start:end]
        # Bei target_only den ganzen Verlauf behalten
        for epoch, r2 in enumerate(val_r2, 1):
            records.append({
                "fraction": frac,
                "method": method,
                "run": run,
                "epoch": epoch,
                "r2": r2
            })
    return pd.DataFrame(records)

df_all = extract_finetune_curves(results)

# Gruppieren
def summarize_group(df):
    return df.groupby(["fraction", "method", "epoch"])["r2"].agg(
        mean="mean",
        q25=lambda x: np.percentile(x, 25),
        q75=lambda x: np.percentile(x, 75)
    ).reset_index()

summary_df = summarize_group(df_all)

# Mapping für method label
summary_df["method"] = summary_df["method"].map({
    "target_only": "Target Only",
    "transfer": "Transfer Learning"
})

# Plot
plt.figure(figsize=(12, 6))

for frac in sample_fractions:
    df_frac = summary_df[summary_df["fraction"] == frac]
    color = color_map[frac]
    for method in ["Transfer Learning", "Target Only"]:
        df_m = df_frac[df_frac["method"] == method]
        linestyle = "-" if method == "Transfer Learning" else "--"
        label = f"{int(frac * 100)}% - {method}"
        plt.plot(df_m["epoch"], df_m["mean"], linestyle=linestyle, label=label, color=color)
        plt.fill_between(df_m["epoch"], df_m["q25"], df_m["q75"], alpha=0.15, color=color)

plt.title("Validation R²: Fine-Tuning vs Target-Only by Train Fraction")
plt.xlabel("Epoch")
plt.ylabel("Validation R²")
plt.ylim(-0.05, 0.65)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
results_ft_transfer = []

for r in range(15):
    feature_space = pd.concat([X_train_reconstructed[shared_columns], X_test_reconstructed[shared_columns]])
    
    # Instantiate the model.
    model = FTTransformerModel(
        sample_df=feature_space,
        seed=r,
        **additional_config,  # Unpack the additional hyperparameters.
        **best_param          # Unpack the best parameter settings.
    )    

    model.train(X_train_reconstructed[shared_columns], y_train, X_test_reconstructed[shared_columns], y_test)

    model.patience = 30
    
    all_cols = list(X_train_reconstructed.columns)
    model.fine_tune(
        X_new_train=X_train_reconstructed[all_cols],
        y_new_train=y_train,
        X_new_val=X_test_reconstructed[all_cols],
        y_new_val=y_test,
        additional_epochs=100,
        freeze_backbone=False,   # try False if you need more capacity
        freeze_shared=False     # shared embeddings WILL adapt
    )

    
    results_ft_transfer.append(model.get_training_history())

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

plt.rcParams["font.family"] = "Times New Roman"

def prepare_plot_data_transfer_reset_finetune(resulting_values):
    pretrain_data = []
    finetune_data = []

    for run_id, run in enumerate(resulting_values):
        val_r2 = run["val_r2"]
        pretrain_end = run["max_train_epoch"]
        finetune_len = run["max_finetune_epoch"]

        val_r2_pre = val_r2[:pretrain_end]
        val_r2_fine = val_r2[pretrain_end : pretrain_end + finetune_len]

        for i, r2 in enumerate(val_r2_pre):
            pretrain_data.append({"epoch": i + 1, "r2": r2, "run": run_id})
        for i, r2 in enumerate(val_r2_fine):
            finetune_data.append({"epoch": i + 1, "r2": r2, "run": run_id})

    return pd.DataFrame(pretrain_data), pd.DataFrame(finetune_data)

# Daten vorbereiten
df_pre, df_fine = prepare_plot_data_transfer_reset_finetune(results_ft_transfer)

# Gruppieren
def summarize(df):
    return df.groupby("epoch")["r2"].agg(
        mean="mean",
        q25=lambda x: np.percentile(x, 25),
        q75=lambda x: np.percentile(x, 75)
    ).reset_index()

summary_pre = summarize(df_pre)
summary_fine = summarize(df_fine)

# Farben & Styles
color_pre = "#00376c"   # blau
color_fine = "#EF7C00"  # orange

# Plot
plt.figure(figsize=(12, 6))

# Pretrain
plt.plot(summary_pre["epoch"], summary_pre["mean"], linestyle="dotted", color=color_pre, linewidth=2, label="Pretrain (36 Features of UP5)")
plt.fill_between(summary_pre["epoch"], summary_pre["q25"], summary_pre["q75"], color=color_pre, alpha=0.2)

# Finetune
plt.plot(summary_fine["epoch"], summary_fine["mean"], linestyle="solid", color=color_fine, linewidth=2, label="Finetune (317 Features of UP5)")
plt.fill_between(summary_fine["epoch"], summary_fine["q25"], summary_fine["q75"], color=color_fine, alpha=0.2)

# Styling
plt.xlabel("Epoch", fontsize=14)
plt.ylabel("Validation R²", fontsize=14)
plt.title("FT-MDNTransformer: Heterogeneous Feature Transfer Learning with UP5 only", fontsize=16)
plt.ylim(-0.05, 0.65)
plt.legend(loc="lower right", fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.savefig(f"ft_mdn_transformer_transfer_up5_only_heterogeneous_{up5_helper.timestamp()}.pdf")
plt.show()


In [ ]:
feature_space = pd.concat([X_train_reconstructed[shared_columns[:15]], X_test_reconstructed[shared_columns[:15]]])

# Instantiate the model.
model = FTTransformerModel(
    sample_df=feature_space,
    **additional_config,  # Unpack the additional hyperparameters.
    **best_param          # Unpack the best parameter settings.
)

In [ ]:
model.train(X_train_reconstructed[shared_columns[:15]], y_train, X_test_reconstructed[shared_columns[:15]], y_test)

In [ ]:
model.patience = 20

all_cols = list(X_train_reconstructed.columns)
model.fine_tune(
    X_new_train=X_train_reconstructed[all_cols],
    y_new_train=y_train,
    X_new_val=X_test_reconstructed[all_cols],
    y_new_val=y_test,
    additional_epochs=60,
    freeze_backbone=False,   # try False if you need more capacity
    freeze_shared=False     # shared embeddings WILL adapt
)


In [ ]:
len(model.get_training_history()["val_r2"])

In [ ]:
model.max_train_epoch

In [ ]:
model.max_finetune_epoch

In [ ]:
len(results_ft_transfer[0]["val_r2"])

In [ ]:
results_ft_transfer[1]["val_r2"]

In [ ]:
end

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

history = results_ft_transfer[0]
val_r2 = history["val_r2"]

# Phase boundaries
split = history["max_train_epoch"] #model.history["max_train_epoch"]
end = history["max_finetune_epoch"] #model.history["max_train_epoch"]

# Validation R² slices
val_r2_pre = val_r2[:split]
val_r2_fine = val_r2[split:split+end]

# Plot
plt.figure(figsize=(10, 5))

# Fine-tuning (solid orange)
plt.plot(range(1, len(val_r2_fine)+1), val_r2_fine, '-', color="#EF7C00", linewidth=2, label='Finetune : Validation R²')

# Pretraining (dotted blue)
plt.plot(range(1, len(val_r2_pre)+1), val_r2_pre, '--', color="#00376c", linewidth=2, label='Pretrain: Validation R²')

# Styling
plt.xlabel('Epoch')
plt.ylabel('R² Score')
plt.title('Validation R² Over Epochs (No Freeze)')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

history = model.get_training_history()
val_r2 = history["val_r2"]

# Phase boundaries
split = model.max_train_epoch #model.history["max_train_epoch"]
end = model.max_finetune_epoch #model.history["max_train_epoch"]

# Validation R² slices
val_r2_pre = val_r2[:split]
val_r2_fine = val_r2[split:end]

# Plot
plt.figure(figsize=(10, 5))

# Fine-tuning (solid orange)
plt.plot(range(1, len(val_r2_fine)+1), val_r2_fine, '-', color="#EF7C00", linewidth=2, label='Finetune : Validation R²')

# Pretraining (dotted blue)
plt.plot(range(1, len(val_r2_pre)+1), val_r2_pre, '--', color="#00376c", linewidth=2, label='Pretrain: Validation R²')

# Styling
plt.xlabel('Epoch')
plt.ylabel('R² Score')
plt.title('Validation R² Over Epochs (No Freeze)')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

history = model.get_training_history()
val_r2 = history["val_r2"]

# Phase boundaries
split = model.max_train_epoch
end = model.max_finetune_epoch

# Validation R² slices
val_r2_pre = val_r2[:split]
val_r2_fine = val_r2[split:end]

# Plot
plt.figure(figsize=(10, 5))

# Fine-tuning (solid orange)
plt.plot(range(1, len(val_r2_fine)+1), val_r2_fine, '-', color="#EF7C00", linewidth=2, label='Finetune : Validation R²')

# Pretraining (dotted blue)
plt.plot(range(1, len(val_r2_pre)+1), val_r2_pre, '--', color="#00376c", linewidth=2, label='Pretrain: Validation R²')

# Styling
plt.xlabel('Epoch')
plt.ylabel('R² Score')
plt.title('Validation R² Over Epochs (Freeze Backbone)')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
model.patience = 30

all_cols = list(X_train_reconstructed.columns)
model.fine_tune(
    X_new_train=X_train_reconstructed[all_cols],
    y_new_train=y_train,
    X_new_val=X_test_reconstructed[all_cols],
    y_new_val=y_test,
    additional_epochs=60,
    freeze_backbone=True,   # try False if you need more capacity
    freeze_shared=False     # shared embeddings WILL adapt
)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

train_jsd = model.get_training_history()["train_jsd"]
val_jsd = model.get_training_history()["val_jsd"]
jsd_max = np.log(2)

# Plot mit benutzerdefinierten Farben
plt.figure(figsize=(10, 5))
plt.plot(train_jsd, label='Train JSD', color="#00376c", linewidth=2)
plt.plot(val_jsd, label='Validation JSD', color="#EF7C00", linewidth=2)
plt.axhline(jsd_max, color="gray", linestyle="--", linewidth=1.5)
plt.text(len(train_jsd)-1.5, jsd_max - 0.035, 'Theoretical Max (ln(2)) ≈ 0.693', 
         verticalalignment='bottom', horizontalalignment='right', color='gray')

plt.xlabel('Epoch')
plt.ylabel('Jensen-Shannon Divergence')
plt.title('Training and Validation JSD over Epochs')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("jsd_plot.pdf")
plt.show()


In [ ]:
results_target_encoded = []

for r in range(15):

    feature_space = pd.concat([X_train_encoded, X_test_encoded])
    
    # Instantiate the model.
    model = FTTransformerModel(
        sample_df=feature_space,
        seed=r,
        **additional_config,  # Unpack the additional hyperparameters.
        **best_param          # Unpack the best parameter settings.
    )

    model.train(X_train_encoded, y_train, X_test_encoded, y_test)
    
    results_target_encoded.append(model.get_training_history())

In [ ]:
results_with_dummies = []

for r in range(15):
    feature_space = pd.concat([X_train, X_test])
    
    # Instantiate the model.
    model = FTTransformerModel(
        sample_df=feature_space,
        seed=r,
        **additional_config,  # Unpack the additional hyperparameters.
        **best_param          # Unpack the best parameter settings.
    )    

    model.train(X_train, y_train, X_test, y_test)
    
    results_with_dummies.append(model.get_training_history())

In [ ]:
results_categorical = []

for r in range(15):
    feature_space = pd.concat([X_train_reconstructed, X_test_reconstructed])
    
    # Instantiate the model.
    model = FTTransformerModel(
        sample_df=feature_space,
        seed=r,
        **additional_config,  # Unpack the additional hyperparameters.
        **best_param          # Unpack the best parameter settings.
    )    

    model.train(X_train_reconstructed, y_train, X_test_reconstructed, y_test)
    
    results_categorical.append(model.get_training_history())

In [ ]:
resulting_values_with_dummies = [entry["val_r2"] for entry in results_with_dummies]
resulting_values_categorical = [entry["val_r2"] for entry in results_categorical]
resulting_values_target_enc = [entry["val_r2"] for entry in results_target_encoded]

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
plt.rcParams["font.family"] = "Times New Roman"


def prepare_plot_data(resulting_values, label):
    data = []
    stopping_points = []
    max_epoch = max(len(run) for run in resulting_values)

    for run_id, run in enumerate(resulting_values):
        for epoch, val in enumerate(run):
            data.append({"epoch": epoch, "r2": val, "model": label})
        if len(run) < max_epoch:
            stopping_points.append({
                "epoch": len(run) - 1,
                "r2": run[-1],
                "model": label
            })
    df = pd.DataFrame(data)
    df_stopping = pd.DataFrame(stopping_points)
    return df, df_stopping

# Prepare data
df_dummy, stop_dummy = prepare_plot_data(resulting_values_with_dummies, "Dummies")
df_cat, stop_cat = prepare_plot_data(resulting_values_categorical, "Categorical")
df_target, stop_target = prepare_plot_data(resulting_values_target_enc, "Target Encoding")

df_all = pd.concat([df_dummy, df_cat, df_target])
df_all.to_csv("df_all_ft_transformer_encoding_comparison_new.csv")
stop_all = pd.concat([stop_dummy, stop_cat, stop_target])

# Compute mean and IQR per epoch
summary = df_all.groupby(["model", "epoch"])["r2"].agg(
    mean="mean",
    q25=lambda x: np.percentile(x, 25),
    q75=lambda x: np.percentile(x, 75)
).reset_index()

# Set custom colors and markers
line_colors = {
    "Dummies": "#00376c",           # dark blue
    "Categorical": "#EF7C00",       # orange
    "Target Encoding": "#2E8B57"    # sea green
}
marker_shapes = {
    "Dummies": "X",
    "Categorical": "^",
    "Target Encoding": "o"
}

# Plot
plt.figure(figsize=(14, 7))

for model in line_colors.keys():
    model_data = summary[summary["model"] == model]
    color = line_colors[model]

    # Count how many runs contributed at each epoch
    counts = df_all[df_all["model"] == model].groupby("epoch")["r2"].count().reset_index(name="count")
    merged = model_data.merge(counts, on="epoch")

    # Split into segments
    segments = []
    current_segment = {"style": None, "x": [], "y": []}

    def get_style(n):
        if n >= 10:
            return "solid"
        elif n >= 6:
            return "dashed"
        elif n >= 3:
            return "dotted"
        else:
            return None

    for _, row in merged.iterrows():
        style = get_style(row["count"])
        if style != current_segment["style"]:
            if current_segment["style"] is not None:
                segments.append(current_segment)
            current_segment = {"style": style, "x": [], "y": []}
        if style is not None:
            current_segment["x"].append(row["epoch"])
            current_segment["y"].append(row["mean"])
    if current_segment["style"] is not None and current_segment["x"]:
        segments.append(current_segment)

    # Plot each segment with its corresponding style
    for seg in segments:
        plt.plot(seg["x"], seg["y"], label=f"{model} (mean)" if seg == segments[0] else None,
                 color=color, linewidth=2,
                 linestyle=seg["style"])

    # IQR band (only for epochs with >=3 runs)
    valid_iqr = merged[merged["count"] >= 3]
    plt.fill_between(valid_iqr["epoch"], valid_iqr["q25"], valid_iqr["q75"], 
                     color=color, alpha=0.2, label=f"{model} IQR")

    
# Final formatting
plt.xlabel("Epoch", fontsize=14)
plt.ylabel("Validation R²",fontsize=14)
#plt.title("FT-MDNTransformer: Feature Encoding Comparison (15 runs each)")
plt.title("FT-MDNTransformer: Encoding Comparison on UP5 (15 runs each)\n"
          "Avg. Runtime: Dummies 39:26, Categorical 15:02, TargetEnc 16:49", fontsize=16)

plt.legend(loc="lower right", fontsize=14)
plt.grid(True)
plt.tight_layout()

# Optional: save to file
plt.savefig(f"../figures/feature_encoding_comparison_ft_{up5_helper.timestamp()}_new.pdf", dpi=300, bbox_inches="tight")

plt.show()



In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def prepare_plot_data(resulting_values, label):
    data = []
    stopping_points = []
    max_epoch = max(len(run) for run in resulting_values)

    for run_id, run in enumerate(resulting_values):
        for epoch, val in enumerate(run):
            data.append({"epoch": epoch, "r2": val, "model": label})
        if len(run) < max_epoch:
            stopping_points.append({
                "epoch": len(run) - 1,
                "r2": run[-1],
                "model": label
            })
    df = pd.DataFrame(data)
    df_stopping = pd.DataFrame(stopping_points)
    return df, df_stopping

# Prepare data
df_dummy, stop_dummy = prepare_plot_data(resulting_values_with_dummies, "Dummies")
df_cat, stop_cat = prepare_plot_data(resulting_values_categorical, "Categorical")
df_all = pd.concat([df_dummy, df_cat])
stop_all = pd.concat([stop_dummy, stop_cat])

# Compute mean and IQR per epoch
summary = df_all.groupby(["model", "epoch"])["r2"].agg(
    mean="mean",
    q25=lambda x: np.percentile(x, 25),
    q75=lambda x: np.percentile(x, 75)
).reset_index()

# Set custom colors
line_colors = {
    "Dummies": "#00376c",   # dark blue
    "Categorical": "#EF7C00"      # orange
}
marker_shapes = {
    "Dummies": "X",
    "Categorical": "^"
}

# Plot
plt.figure(figsize=(14, 7))

for model in ["Dummies", "Categorical"]:
    model_data = summary[summary["model"] == model]
    color = line_colors[model]
    
    # Mean line
    plt.plot(model_data["epoch"], model_data["mean"], label=f"{model} (mean)", color=color, linewidth=2)
    
    # IQR band
    plt.fill_between(model_data["epoch"], model_data["q25"], model_data["q75"], 
                     color=color, alpha=0.2, label=f"{model} IQR")

# Early stopping markers
for model in ["Dummies", "Categorical"]:
    subset = stop_all[stop_all["model"] == model]
    if not subset.empty:
        plt.scatter(subset["epoch"], subset["r2"], 
                    color=line_colors[model], marker=marker_shapes[model], 
                    s=50, label=f"{model} Early Stop")

# Final formatting
plt.xlabel("Epoch")
plt.ylabel("Validation R²")
plt.title("FT-MDNTransformer: Dummy vs. Categorical Features with IQR and Early Stopping (15 runs each)")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.savefig(f"../figures/up5_dummy_vs_category_ft_mdn_comparison_{up5_helper.timestamp()}.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
results_with_dummies[2].keys()

In [ ]:
best_vals = pd.DataFrame([
    {"model": "Dummies", "best_val_r2": run["best_val_r2"]} for run in results_with_dummies
] + [
    {"model": "Categorical", "best_val_r2": run["best_val_r2"]} for run in results_categorical
])

sns.boxplot(x="model", y="best_val_r2", data=best_vals)
plt.title("Best Validation R² Distribution")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def extract_loss_curves(results, model_label):
    all_train_loss = []
    all_val_loss = []

    for run in results:
        train_loss = run["train_loss"]
        val_loss = run["val_loss"]
        epochs = len(train_loss)

        for epoch in range(epochs):
            all_train_loss.append({
                "epoch": epoch,
                "loss": train_loss[epoch],
                "type": "train",
                "model": model_label
            })
            all_val_loss.append({
                "epoch": epoch,
                "loss": val_loss[epoch],
                "type": "val",
                "model": model_label
            })

    return pd.DataFrame(all_train_loss + all_val_loss)

# Create loss dataframes
df_loss_dummies = extract_loss_curves(results_with_dummies, "Dummies")
df_loss_cat = extract_loss_curves(results_categorical, "Categorical")
df_loss = pd.concat([df_loss_dummies, df_loss_cat])

# Compute mean and IQR for smoothing
summary_loss = df_loss.groupby(["model", "type", "epoch"])["loss"].agg(
    mean="mean",
    q25=lambda x: np.percentile(x, 25),
    q75=lambda x: np.percentile(x, 75)
).reset_index()

# Plot
plt.figure(figsize=(14, 7))
color_map = {
    ("Dummies", "train"): "#00376c",
    ("Dummies", "val"): "#00376c",
    ("Categorical", "train"): "#EF7C00",
    ("Categorical", "val"): "#EF7C00"
}
linestyle_map = {"train": "--", "val": "-"}

for (model, typ), group in summary_loss.groupby(["model", "type"]):
    c = color_map[(model, typ)]
    ls = linestyle_map[typ]
    label = f"{model} {typ}"
    
    plt.plot(group["epoch"], group["mean"], label=label, color=c, linestyle=ls, linewidth=2)
    plt.fill_between(group["epoch"], group["q25"], group["q75"], color=c, alpha=0.15)

plt.title("Training and Validation Loss Curves with IQR")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(loc="upper right")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
p

In [ ]:
resulting_values = [entry["val_r2"] for entry in results_with_dummies]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Suppose: resulting_values = [[...], [...], ...] — each list may be of different length
data = []
stopping_points = []

# Find the longest run (i.e., max number of epochs)
max_epoch = max(len(run) for run in resulting_values)

for run_id, run in enumerate(resulting_values):
    for epoch, val in enumerate(run):
        data.append({"epoch": epoch, "r2": val})
    if len(run) < max_epoch:
        # Only count as early stopping if not a full-length run
        stopping_points.append({
            "epoch": len(run) - 1,
            "r2": run[-1],
            "run_id": run_id
        })

df = pd.DataFrame(data)
df_stopping = pd.DataFrame(stopping_points)

# Compute average R² per epoch
avg_r2_per_epoch = df.groupby("epoch")["r2"].mean()

# Create the plot
plt.figure(figsize=(12, 6))
sns.boxplot(x="epoch", y="r2", data=df, color="lightgray", showfliers=False)
plt.plot(avg_r2_per_epoch.index, avg_r2_per_epoch.values, label="Average R²", color="blue", linewidth=2)
plt.scatter(avg_r2_per_epoch.index, avg_r2_per_epoch.values, color="blue", s=30)

# Only show early stopping markers for incomplete runs
if not df_stopping.empty:
    plt.scatter(df_stopping["epoch"], df_stopping["r2"], 
                color="red", marker="X", s=80, label="Early Stopping")

plt.xlabel("Epoch")
plt.ylabel("Validation R²")
plt.title("R² over Epochs with Early Stopping Indicators")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
model.best_epoch

In [ ]:
model.best_val_r2

In [ ]:
model.get_training_history()["val_r2"]

In [ ]:
model.

In [ ]:
model.fine_tune(X_train, y_train, X_test, y_test, additional_epochs=5)

In [ ]:
model.train(X_train, y_train, X_test, y_test)

In [ ]:
def plot_training_history(model):
    """
    Generate plots for training history:
      1) Training vs. Validation Loss
      2) Training vs. Validation R²
      3) Combined Mixture Component Means
      4) Learned Temperature over Epochs

    Args:
      model: Trained FTTransformerModel instance
    """
    history = model.get_training_history()
    epochs = list(range(1, len(history['train_loss']) + 1))

    import matplotlib.pyplot as plt
    import numpy as np

    # 1) Train vs Val Loss
    plt.figure()
    plt.plot(epochs, history['train_loss'], label='Train Loss')
    plt.plot(epochs, history['val_loss'],   label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.show()

    # 2) Train vs Val R²
    plt.figure()
    plt.plot(epochs, history['train_r2'], label='Train R²')
    plt.plot(epochs, history['val_r2'],   label='Val R²')
    plt.xlabel('Epoch')
    plt.ylabel('R² Score')
    plt.title('Training and Validation R² Scores')
    plt.legend()
    plt.show()

    # 3) Combined Mixture Component Means
    means = np.array(history['val_means'])  # shape: (epochs, n_components)
    plt.figure()
    for idx in range(means.shape[1]):
        plt.plot(epochs, means[:, idx], label=f'Component {idx} Mean')
    plt.xlabel('Epoch')
    plt.ylabel('Mean Value')
    plt.title('Mixture Component Means over Epochs')
    plt.legend()
    plt.show()

    # 4) Learned Temperature
    plt.figure()
    plt.plot(epochs, history['temperature_history'], label='Temperature')
    plt.xlabel('Epoch')
    plt.ylabel('Temperature')
    plt.title('Learned Temperature over Epochs')
    plt.legend()
    plt.show()

plot_training_history(model)

In [ ]:
trained_bias = model.model.head.fc2.bias.data.clone()
trained_weight = model.model.head.fc2.weight.data.clone()


In [ ]:
trained_bias

In [ ]:
trained_weight

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

def plot_mdn_prediction(alpha, mu, sigma, y_true=None):
    x = np.linspace(0, 1.2, 500)
    total_pdf = np.zeros_like(x)
    for a, m, s in zip(alpha, mu, sigma):
        pdf = a * (1 / (np.sqrt(2 * np.pi) * s)) * np.exp(-0.5 * ((x - m) / s)**2)
        total_pdf += pdf
        plt.plot(x, pdf, linestyle='--', alpha=0.5)
    plt.plot(x, total_pdf, label='Mixture PDF', linewidth=2)
    if y_true is not None:
        plt.axvline(y_true, color='red', linestyle=':', label='True y')
    plt.title("Predicted Mixture Distribution")
    plt.xlabel("y")
    plt.ylabel("Density")
    plt.legend()
    plt.grid(True)
    plt.show()

def get_mdn_params_for_sample(model, X_row):
    model.model.eval()
    X_numeric = torch.tensor(
        X_row[model.numeric_columns].to_numpy(dtype=np.float32).reshape(1, -1)
    ).to(model.device)
    X_cat = torch.tensor(
        X_row[model.categorical_columns].to_numpy(dtype=np.int64).reshape(1, -1)
    ).to(model.device)
    with torch.no_grad():
        _, (alpha, mu, sigma) = model.model(X_numeric, X_cat)
    return alpha[0].cpu().numpy(), mu[0].cpu().numpy(), sigma[0].cpu().numpy()

i=0

In [ ]:

X_row = X_test.iloc[i]
y_true = y_test[i]

alpha, mu, sigma = get_mdn_params_for_sample(model, X_row)
plot_mdn_prediction(alpha, mu, sigma, y_true=y_true)
i += 1 

In [ ]:
from sklearn.metrics import r2_score

y_pred_mean = model.predict(X_test)
y_pred_mode = model.predict(X_test, use_mode=True)

r2_mean = r2_score(y_test, y_pred_mean)
r2_mode = r2_score(y_test, y_pred_mode)

print(f"R² (mean prediction): {r2_mean:.4f}")
print(f"R² (mode prediction): {r2_mode:.4f}")


import matplotlib.pyplot as plt

plt.hist(y_pred_mean, bins=40, alpha=0.5, label="Mean prediction")
plt.hist(y_pred_mode, bins=40, alpha=0.5, label="Mode prediction")
plt.hist(y_test, bins=40, histtype='step', edgecolor='black', linewidth=1.5, label="Truth")
plt.legend()
plt.title(f"Prediction Distributions\nR² Mean = {r2_mean:.2f}, R² Mode = {r2_mode:.2f}")
plt.grid(True)
plt.show()



In [ ]:
# Retrieve results (assuming best_model_results returns these values).
r2_val, r2_val_train , epoch, y_true_train, y_predict_train, y_true_test, y_predict_test = model.best_model_results
print(f"\n ------->>>> Epoch {epoch}: R² = {r2_val}  (Train R² = {r2_val_train})")

# Plot training and evaluation histograms.
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].hist(y_true_train, bins=30, alpha=0.6, label="Actual")
axs[0].hist(y_predict_train, bins=30, alpha=0.6, label="Predicted")
axs[0].set_title("Train Data")
axs[0].legend()
axs[1].hist(y_true_test, bins=30, alpha=0.6, label="Actual")
axs[1].hist(y_predict_test, bins=30, alpha=0.6, label="Predicted")
axs[1].set_title("Eval Data")
axs[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
def plot_mdn_prediction(alpha, mu, sigma, y_true=None):
    x = np.linspace(0, 1.2, 500)
    total_pdf = np.zeros_like(x)
    for a, m, s in zip(alpha, mu, sigma):
        pdf = a * (1 / (np.sqrt(2 * np.pi) * s)) * np.exp(-0.5 * ((x - m) / s)**2)
        total_pdf += pdf
        plt.plot(x, pdf, linestyle='--', alpha=0.5)  # individual components
    plt.plot(x, total_pdf, label='Mixture PDF', linewidth=2)
    if y_true is not None:
        plt.axvline(y_true, color='red', linestyle=':', label='True y')
    plt.title("Predicted Mixture Distribution")
    plt.xlabel("y")
    plt.ylabel("Density")
    plt.legend()
    plt.grid(True)
    plt.show()

plot_mdn_prediction([0.2, 0.3, 0.5], [0.2, 0.4, 0.9], 	[0.2, 0.2, 0.1], 0.8)

In [ ]:
model.train(X_train, y_train, X_test, y_test)

# Optionally, to check performance after training:
print("Train R²:", model.compute_r2(X_train, y_train))
print("Test R²:", model.compute_r2(X_test, y_test))
